# Temporal rule-based classifier

This runs a small labelled set of **whole videos** and checks the bottle-angle timeline.

- `normal`: upright and never persistently flat.
- `fallen_on_entry`: first visible states are tilted/flat with no earlier upright state.
- `fell_during_view`: upright is followed by tilted/flat in the same video.

Add a few normal clips to `stoppage_detection_and_classification/cnn_classifier/data/training_clips/normal/` before running the test.


In [ ]:
from pathlib import Path
import json
import sys

# Find the project folder before importing the local VideoModule package.
project_root = Path.cwd().resolve()
if project_root.name == "classifier":
    project_root = project_root.parent
if not (project_root / "VideoModule").exists():
    project_root = project_root / "24H_Insights"
if not (project_root / "VideoModule").exists():
    raise FileNotFoundError("Could not find the 24H_Insights project folder")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Use VideoModule for NCC keyframes, video decoding, sampling, and smoothing.
from VideoModule.embeddings.frame_embeddings import make_embedder
from VideoModule.image_similarity import Roi, crop_to_roi, find_similarity_threshold, score_frames_against_references
from VideoModule.io import discover_video_clips, read_video
from VideoModule.preprocessing import sample_fps, smooth_signal

In [ ]:
# Start with two clips per class so the result is quick to inspect.
CLIPS_PER_CLASS = 2
TARGET_FPS = 20.0
# Upright bottles have this non-zero long-axis angle in the camera view.
NORMAL_ANGLE_DEG = 20.0

# Upright ranges are offsets added to NORMAL_ANGLE_DEG: 20 + (-3, 2) = 17 to 22 degrees.
UPRIGHT_TILT_RANGES_DEG = [(-3.5, 3.5)]
UPRIGHT_ANGLE_RANGES_DEG = [
    (NORMAL_ANGLE_DEG + lower_deg, NORMAL_ANGLE_DEG + upper_deg)
    for lower_deg, upper_deg in UPRIGHT_TILT_RANGES_DEG
]

# Tilted and flat ranges are absolute normalized minAreaRect angles.
TILTED_ANGLE_RANGES_DEG = [
    (-55.0, 16.5),  # Backward tilt.
    (23.51, 55.0),    # Forward tilt.
]

FLAT_ANGLE_RANGES_DEG = [
    (-90.0, -55.0),  # Fallen backward.
    (55.0, 90.0),    # Fallen forward.
]

MIN_STATE_FRAMES = 4
MIN_VISIBLE_FRACTION = 0.20
ENTRY_WINDOW_SECONDS = 1.25

# NCC settings select which sampled frames are exported as keyframes.
NCC_SIZE = 128
NCC_MIN_SIMILARITY = 0.55

NCC_KEYFRAME_DISTANCE_SECONDS = 0.50

# Accept minAreaRect candidates only within this absolute long-side pixel range.
MIN_RECTANGLE_LENGTH_PX = 100.0
MAX_RECTANGLE_LENGTH_PX = 142.5


def value_is_in_ranges(value_deg, ranges_deg):
    # Treat both ends of each configured range as inclusive.
    return any(lower_deg <= value_deg <= upper_deg for lower_deg, upper_deg in ranges_deg)


def normalise_min_area_rect_angle(rectangle_width, rectangle_height, raw_angle_deg):
    # Convert OpenCV's rectangle angle to the long-axis angle first.
    long_axis_angle_deg = float(raw_angle_deg)
    if rectangle_width < rectangle_height:
        long_axis_angle_deg += 90.0
    if long_axis_angle_deg > 90.0:
        long_axis_angle_deg -= 180.0

    # OpenCV image coordinates report this camera tilt with the opposite sign.
    return -long_axis_angle_deg


def calculate_signed_tilt(angle_deg):
    # Report deviation from the non-zero normal angle for diagnostics.
    return ((angle_deg - NORMAL_ANGLE_DEG + 90.0) % 180.0) - 90.0


def classify_angle(angle_deg):
    # Upright takes priority where its absolute angle band shares a boundary.
    if value_is_in_ranges(angle_deg, UPRIGHT_ANGLE_RANGES_DEG):
        return "upright"

    # The remaining configured ranges are absolute minAreaRect angles.
    if value_is_in_ranges(angle_deg, FLAT_ANGLE_RANGES_DEG):
        return "flat"
    if value_is_in_ranges(angle_deg, TILTED_ANGLE_RANGES_DEG):
        return "tilted"
    return "uncertain"


def classify_tilt(signed_tilt_deg):
    # Convert a normal-relative tilt back to its absolute camera angle.
    angle_deg = ((NORMAL_ANGLE_DEG + signed_tilt_deg + 90.0) % 180.0) - 90.0
    return classify_angle(angle_deg)

pipeline_root = project_root / "stoppage_detection_and_classification"
cnn_classifier_dir = pipeline_root / "cnn_classifier"
test_clips_path = cnn_classifier_dir / "data" / "training_clips"
output_folder = cnn_classifier_dir / "output" / "05_evaluate_temporal_rule_based_classifier"
if not test_clips_path.is_dir():
    raise FileNotFoundError(f"Test clips folder does not exist: {test_clips_path}")
output_folder.mkdir(parents=True, exist_ok=True)

# Load only the classifier-specific Pitwall mask and ROI configuration.
bottle_config_path = project_root / "measurements" / "pitwall_config" / "bottles_classifier.json"
with bottle_config_path.open(encoding="utf-8") as file:
    bottle_config = json.load(file)


def validate_bottle_classifier_config(config):
    # Require an enabled rectangular include ROI; never fall back to another ROI.
    roi_settings = config.get("roi", {})
    if not roi_settings.get("enabled", False):
        raise ValueError("bottles_classifier.json must have ROI masking enabled")

    enabled_rois = [
        roi_record
        for roi_record in roi_settings.get("active_rois", [])
        if roi_record.get("enabled", True)
    ]
    include_rois = [roi_record for roi_record in enabled_rois if roi_record.get("mode", "include") == "include"]
    if len(include_rois) != 1:
        raise ValueError("NCC scoring requires exactly one enabled include ROI in bottles_classifier.json")
    if any(roi_record.get("shape", "rectangle") != "rectangle" for roi_record in enabled_rois):
        raise ValueError("The classifier currently requires rectangular ROIs")

    # Reject enabled processors that this notebook cannot reproduce exactly.
    processor_order = config.get("app", {}).get("mask_processor_order", [])
    supported_processors = {"hue_detection", "contour_filtering", "morphology"}
    for processor_name in processor_order:
        processor_settings = config.get(processor_name, {})
        if processor_settings.get("enabled", False) and processor_name not in supported_processors:
            raise ValueError(f"Unsupported enabled mask processor: {processor_name}")

    # Every supported enabled processor must appear in the saved Pitwall order.
    for processor_name in supported_processors:
        if config.get(processor_name, {}).get("enabled", False) and processor_name not in processor_order:
            raise ValueError(f"Enabled processor is missing from mask_processor_order: {processor_name}")


validate_bottle_classifier_config(bottle_config)


def create_bottle_roi_mask(frame_shape, config):
    # Start with a black mask and enable the configured include regions.
    frame_height, frame_width = frame_shape[:2]
    roi_settings = config.get("roi", {})
    if not roi_settings.get("enabled", False):
        return np.full((frame_height, frame_width), 255, dtype=np.uint8)

    enabled_rois = [
        roi_record
        for roi_record in roi_settings.get("active_rois", [])
        if roi_record.get("enabled", True)
    ]
    if not enabled_rois:
        return np.full((frame_height, frame_width), 255, dtype=np.uint8)

    roi_mask = np.zeros((frame_height, frame_width), dtype=np.uint8)
    for roi_record in enabled_rois:
        if roi_record.get("mode", "include") != "include":
            continue
        x1 = max(0, int(roi_record.get("x", 0)))
        y1 = max(0, int(roi_record.get("y", 0)))
        x2 = min(frame_width, x1 + max(0, int(roi_record.get("width", 0))))
        y2 = min(frame_height, y1 + max(0, int(roi_record.get("height", 0))))
        roi_mask[y1:y2, x1:x2] = 255

    # An exclusion-only configuration starts from the full frame.
    if not roi_mask.any():
        roi_mask[:, :] = 255

    for roi_record in enabled_rois:
        if roi_record.get("mode", "include") != "exclude":
            continue
        x1 = max(0, int(roi_record.get("x", 0)))
        y1 = max(0, int(roi_record.get("y", 0)))
        x2 = min(frame_width, x1 + max(0, int(roi_record.get("width", 0))))
        y2 = min(frame_height, y1 + max(0, int(roi_record.get("height", 0))))
        roi_mask[y1:y2, x1:x2] = 0
    return roi_mask


def create_bottle_hsv_mask(frame, hue_settings):
    # Apply the explicit HSV limits exported by Pitwall.
    hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    hue_min = max(0, min(179, int(hue_settings.get("hue_min", 0))))
    hue_max = max(0, min(179, int(hue_settings.get("hue_max", 179))))
    saturation_min = max(0, min(255, int(hue_settings.get("saturation_min", 0))))
    saturation_max = max(0, min(255, int(hue_settings.get("saturation_max", 255))))
    value_min = max(0, min(255, int(hue_settings.get("value_min", 0))))
    value_max = max(0, min(255, int(hue_settings.get("value_max", 255))))

    if hue_min <= hue_max:
        lower_bound = np.array([hue_min, saturation_min, value_min], dtype=np.uint8)
        upper_bound = np.array([hue_max, saturation_max, value_max], dtype=np.uint8)
        return cv2.inRange(hsv_frame, lower_bound, upper_bound)

    # Handle hue ranges that cross OpenCV's 179-to-0 boundary.
    lower_mask = cv2.inRange(
        hsv_frame,
        np.array([hue_min, saturation_min, value_min], dtype=np.uint8),
        np.array([179, saturation_max, value_max], dtype=np.uint8),
    )
    upper_mask = cv2.inRange(
        hsv_frame,
        np.array([0, saturation_min, value_min], dtype=np.uint8),
        np.array([hue_max, saturation_max, value_max], dtype=np.uint8),
    )
    return cv2.bitwise_or(lower_mask, upper_mask)


def contour_passes_bottle_config(contour, contour_settings):
    # Check area and perimeter against the exported limits.
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    if not float(contour_settings.get("min_area", 0)) <= area <= float(contour_settings.get("max_area", np.inf)):
        return False
    if not float(contour_settings.get("min_perimeter", 0)) <= perimeter <= float(contour_settings.get("max_perimeter", np.inf)):
        return False

    # Match Pitwall's mask filter exactly with its axis-aligned bounding rectangle.
    _, _, rectangle_width, rectangle_height = cv2.boundingRect(contour)
    if rectangle_height <= 0:
        return False
    aspect_ratio = float(rectangle_width) / float(rectangle_height)
    if not float(contour_settings.get("min_aspect_ratio", 0)) <= aspect_ratio <= float(contour_settings.get("max_aspect_ratio", np.inf)):
        return False

    # Apply the saved circularity and convexity limits.
    circularity = (4.0 * np.pi * area) / (perimeter ** 2) if perimeter else 0.0
    if not float(contour_settings.get("min_circularity", 0)) <= circularity <= float(contour_settings.get("max_circularity", np.inf)):
        return False
    hull_area = cv2.contourArea(cv2.convexHull(contour))
    convexity = area / hull_area if hull_area else 0.0
    return float(contour_settings.get("min_convexity", 0)) <= convexity <= float(contour_settings.get("max_convexity", np.inf))


def filter_bottle_mask_contours(mask, contour_settings):
    # Retain only contours accepted by bottles_classifier.json.
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filtered_mask = np.zeros_like(mask)
    for contour in contours:
        if contour_passes_bottle_config(contour, contour_settings):
            cv2.drawContours(filtered_mask, [contour], -1, 255, thickness=cv2.FILLED)
    return filtered_mask


def apply_bottle_morphology(mask, morphology_settings):
    # Apply the configured operation, kernel, and iteration count.
    kernel_size = max(1, int(morphology_settings.get("kernel_size", 1)))
    if kernel_size % 2 == 0:
        kernel_size += 1
    iterations = max(1, int(morphology_settings.get("iterations", 1)))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    operation = morphology_settings.get("operation", "open")

    if operation == "erode":
        return cv2.erode(mask, kernel, iterations=iterations)
    if operation == "dilate":
        return cv2.dilate(mask, kernel, iterations=iterations)
    if operation in {"close", "closing"}:
        return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=iterations)
    return cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=iterations)


def build_bottle_mask(frame, config):
    # Start with the configured ROI and follow Pitwall's saved processor order.
    roi_mask = create_bottle_roi_mask(frame.shape, config)
    current_mask = roi_mask.copy()
    processor_order = config.get("app", {}).get("mask_processor_order", [])

    for processor_name in processor_order:
        if processor_name == "hue_detection" and config.get("hue_detection", {}).get("enabled", False):
            hue_mask = create_bottle_hsv_mask(frame, config["hue_detection"])
            current_mask = cv2.bitwise_and(current_mask, hue_mask)
        elif processor_name == "contour_filtering" and config.get("contour_filtering", {}).get("enabled", False):
            current_mask = filter_bottle_mask_contours(current_mask, config["contour_filtering"])
        elif processor_name == "morphology" and config.get("morphology", {}).get("enabled", False):
            current_mask = apply_bottle_morphology(current_mask, config["morphology"])

        # Prevent any processor from expanding beyond the configured bottle ROI.
        current_mask = cv2.bitwise_and(current_mask, roi_mask)

    _, binary_mask = cv2.threshold(current_mask, 0, 255, cv2.THRESH_BINARY)
    return binary_mask


# Use only upright and forward-fallen references for the current classifier.
ncc_reference_folder = (
    cnn_classifier_dir
    / "data"
    / "ncc_reference_frames"
    / "ncc_references_26_06_18-26_06_20"
)
ncc_reference_paths = [
    ncc_reference_folder / "upright.png",
    ncc_reference_folder / "fallen_forward_entry.png",
]
missing_reference_paths = [path for path in ncc_reference_paths if not path.exists()]
if missing_reference_paths:
    raise FileNotFoundError(f"Missing NCC reference images: {missing_reference_paths}")

# Derive the NCC crop directly from the single enabled classifier ROI.
ncc_roi_record = next(
    roi_record
    for roi_record in bottle_config["roi"]["active_rois"]
    if roi_record.get("enabled", True) and roi_record.get("mode", "include") == "include"
)
ncc_roi = Roi(
    x=int(ncc_roi_record["x"]),
    y=int(ncc_roi_record["y"]),
    w=int(ncc_roi_record["width"]),
    h=int(ncc_roi_record["height"]),
)

# Build one two-reference NCC bank. No backward-fall reference is loaded.
ncc_reference_images = []
for reference_path in ncc_reference_paths:
    reference_image = cv2.imread(str(reference_path), cv2.IMREAD_COLOR)
    if reference_image is None:
        raise ValueError(f"Could not decode NCC reference: {reference_path}")
    ncc_reference_images.append(crop_to_roi(reference_image, ncc_roi))

ncc_embedder = make_embedder("ncc", device="cpu", ncc_size=NCC_SIZE)
ncc_reference_embeddings = ncc_embedder.embed_frames(ncc_reference_images)
ncc_reference_names = [path.stem for path in ncc_reference_paths]

print(
    f"Accepted minAreaRect length: {MIN_RECTANGLE_LENGTH_PX:.2f}px "
    f"to {MAX_RECTANGLE_LENGTH_PX:.2f}px"
)


def score_sampled_frames_with_ncc(sampled_frames):
    # Embed in small batches to avoid making one large temporary grayscale array.
    all_scores = []
    all_nearest_references = []
    batch_size = 256
    for batch_start in range(0, len(sampled_frames), batch_size):
        frame_batch = sampled_frames[batch_start:batch_start + batch_size]
        cropped_batch = [crop_to_roi(frame, ncc_roi) for frame in frame_batch]
        frame_embeddings = ncc_embedder.embed_frames(cropped_batch)
        batch_scores, batch_nearest = score_frames_against_references(
            frame_embeddings,
            ncc_reference_embeddings,
            reduce="max",
        )
        all_scores.append(batch_scores)
        all_nearest_references.append(batch_nearest)

    if not all_scores:
        return np.zeros(0, dtype=np.float32), np.zeros(0, dtype=np.int32)
    return np.concatenate(all_scores), np.concatenate(all_nearest_references)


test_folders = {
    "normal": test_clips_path / "normal",
    "fallen_on_entry": test_clips_path / "fallen_before_entry",
    "fell_during_view": test_clips_path / "fallen_in_view",
}

# Select the same small test set each time.
test_videos = []
for expected_label, folder in test_folders.items():
    videos = discover_video_clips(folder, max_clips=CLIPS_PER_CLASS)
    if not videos:
        print(f"No clips found for {expected_label}: {folder}")
    for video_path in videos:
        test_videos.append((expected_label, video_path))

print(f"Selected {len(test_videos)} clips")
pd.DataFrame(test_videos, columns=["expected_label", "video_path"])


In [ ]:
# Extract NCC keyframes first, then calculate tilt only from the saved keyframe images.
results = []
timelines = {}

for video_number, (expected_label, video_path) in enumerate(test_videos, start=1):
    print(f"[{video_number}/{len(test_videos)}] {expected_label}: {video_path.name}")
    frames, original_fps, timestamps = read_video(video_path, return_timestamps=True)
    sampled_indices = sample_fps(len(frames), original_fps, TARGET_FPS)
    sampled_frames = [frames[int(frame_index)] for frame_index in sampled_indices]

    # NCC is the only operation applied across the sampled video.
    ncc_scores, ncc_nearest_indices = score_sampled_frames_with_ncc(sampled_frames)
    keyframe_distance = max(1, int(round(NCC_KEYFRAME_DISTANCE_SECONDS * TARGET_FPS)))
    ncc_keyframe_positions = find_similarity_threshold(
        ncc_scores,
        min_sim=NCC_MIN_SIMILARITY,
        min_distance=keyframe_distance,
    )

    # Export selected frames before running any mask or tilt calculation.
    clip_output_folder = output_folder / expected_label / video_path.stem
    ncc_keyframe_folder = clip_output_folder / "ncc_keyframes"
    ncc_keyframe_folder.mkdir(parents=True, exist_ok=True)
    keyframe_paths_by_position = {}

    for keyframe_number, keyframe_position in enumerate(ncc_keyframe_positions, start=1):
        keyframe_position = int(keyframe_position)
        source_frame_index = int(sampled_indices[keyframe_position])
        reference_name = ncc_reference_names[int(ncc_nearest_indices[keyframe_position])]
        score = float(ncc_scores[keyframe_position])
        keyframe_path = ncc_keyframe_folder / (
            f"{keyframe_number:03d}_frame_{source_frame_index:06d}_{reference_name}_ncc_{score:.3f}.jpg"
        )
        if not cv2.imwrite(str(keyframe_path), frames[source_frame_index]):
            raise RuntimeError(f"Could not export NCC keyframe: {keyframe_path}")
        keyframe_paths_by_position[keyframe_position] = keyframe_path

    # Release the video. Everything below reads only extracted keyframe JPGs.
    del sampled_frames
    del frames

    timeline_rows = []
    for keyframe_position in ncc_keyframe_positions:
        keyframe_position = int(keyframe_position)
        frame_index = int(sampled_indices[keyframe_position])
        keyframe_path = keyframe_paths_by_position[keyframe_position]
        keyframe = cv2.imread(str(keyframe_path), cv2.IMREAD_COLOR)
        if keyframe is None:
            raise RuntimeError(f"Could not read extracted NCC keyframe: {keyframe_path}")

        # Apply the strict bottles_classifier.json mask only to this keyframe.
        bottle_mask = build_bottle_mask(keyframe, bottle_config)
        bottle_contours, _ = cv2.findContours(
            bottle_mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE,
        )

        candidates = []
        for contour in bottle_contours:
            rectangle = cv2.minAreaRect(contour)
            rectangle_width, rectangle_height = rectangle[1]
            if rectangle_width <= 1 or rectangle_height <= 1:
                continue

            # Reject short fragments and merged masks outside the configured length range.
            rectangle_length_px = max(rectangle_width, rectangle_height)
            if not MIN_RECTANGLE_LENGTH_PX <= rectangle_length_px <= MAX_RECTANGLE_LENGTH_PX:
                continue

            # Convert the OpenCV rectangle angle into a consistent long-axis angle.
            angle_deg = normalise_min_area_rect_angle(
                rectangle_width,
                rectangle_height,
                rectangle[2],
            )

            # Keep the tilt sign so the two fall directions remain distinguishable.
            signed_tilt_deg = calculate_signed_tilt(angle_deg)
            absolute_tilt_deg = abs(signed_tilt_deg)
            candidates.append((absolute_tilt_deg, angle_deg, signed_tilt_deg, rectangle_length_px))

        # One fallen bottle is enough, so retain the most tilted candidate.
        selected = max(candidates, key=lambda candidate: candidate[0]) if candidates else None
        time_seconds = timestamps[frame_index] if len(timestamps) > frame_index else frame_index / original_fps
        timeline_rows.append({
            "frame_index": frame_index,
            "time_seconds": float(time_seconds),
            "keyframe_path": str(keyframe_path),
            "ncc_score": float(ncc_scores[keyframe_position]),
            "ncc_smoothed_score": float(ncc_scores[keyframe_position]),
            "nearest_ncc_reference": ncc_reference_names[int(ncc_nearest_indices[keyframe_position])],
            "ncc_in_view": True,
            "ncc_keyframe": True,
            "candidate_count": len(candidates),
            "tilt_deg": np.nan if selected is None else selected[0],
            "angle_deg": np.nan if selected is None else selected[1],
            "signed_tilt_deg": np.nan if selected is None else selected[2],
            "rectangle_length_px": np.nan if selected is None else selected[3],
        })

    # Keep an explicit schema so clips with no keyframes remain valid.
    timeline_columns = [
        "frame_index",
        "time_seconds",
        "keyframe_path",
        "ncc_score",
        "ncc_smoothed_score",
        "nearest_ncc_reference",
        "ncc_in_view",
        "ncc_keyframe",
        "candidate_count",
        "tilt_deg",
        "angle_deg",
        "signed_tilt_deg",
        "rectangle_length_px",
    ]
    timeline = pd.DataFrame(timeline_rows, columns=timeline_columns)
    timeline["detected"] = timeline["tilt_deg"].notna()

    # Smooth only the chronological sequence of extracted keyframe measurements.
    if timeline["detected"].any():
        filled_signed_tilt = timeline["signed_tilt_deg"].interpolate(limit_direction="both").to_numpy()
        smoothing_window = min(5, len(filled_signed_tilt))
        timeline["smoothed_signed_tilt_deg"] = smooth_signal(
            filled_signed_tilt,
            method="moving_average",
            window_size=smoothing_window,
        )
        timeline["smoothed_tilt_deg"] = timeline["smoothed_signed_tilt_deg"].abs()
        timeline["smoothed_angle_deg"] = (
            (NORMAL_ANGLE_DEG + timeline["smoothed_signed_tilt_deg"] + 90.0) % 180.0
        ) - 90.0
    else:
        timeline["smoothed_signed_tilt_deg"] = np.nan
        timeline["smoothed_tilt_deg"] = np.nan
        timeline["smoothed_angle_deg"] = np.nan

    timeline["state"] = "uncertain"
    detected_rows = timeline["detected"]
    timeline.loc[detected_rows, "state"] = timeline.loc[
        detected_rows,
        "smoothed_angle_deg",
    ].apply(classify_angle)

    # Require consecutive keyframes before accepting an upright or flat state.
    persistent_upright = (timeline["state"] == "upright").rolling(MIN_STATE_FRAMES).sum() >= MIN_STATE_FRAMES
    persistent_flat = (timeline["state"] == "flat").rolling(MIN_STATE_FRAMES).sum() >= MIN_STATE_FRAMES
    upright_run_ends = np.flatnonzero(persistent_upright.to_numpy())
    flat_run_ends = np.flatnonzero(persistent_flat.to_numpy())

    # Classify the temporal sequence represented by the NCC keyframes.
    predicted_label = "uncertain"
    transition_position = None
    visible_fraction = float(timeline["detected"].mean()) if len(timeline) else 0.0
    if len(timeline) and visible_fraction >= MIN_VISIBLE_FRACTION:
        if len(flat_run_ends) == 0 and len(upright_run_ends) > 0:
            predicted_label = "normal"
        elif len(flat_run_ends) > 0:
            transition_position = max(0, int(flat_run_ends[0]) - MIN_STATE_FRAMES + 1)
            upright_before_flat = (
                timeline.iloc[:transition_position]["state"] == "upright"
            ).sum() >= MIN_STATE_FRAMES
            if upright_before_flat:
                predicted_label = "fell_during_view"
            else:
                first_visible_position = int(np.flatnonzero(timeline["detected"].to_numpy())[0])
                time_to_flat = (
                    timeline.iloc[transition_position]["time_seconds"]
                    - timeline.iloc[first_visible_position]["time_seconds"]
                )
                if time_to_flat <= ENTRY_WINDOW_SECONDS:
                    predicted_label = "fallen_on_entry"

    transition_time = np.nan if transition_position is None else timeline.iloc[transition_position]["time_seconds"]
    tilted_before_flat = False if transition_position is None else (
        timeline.iloc[:transition_position]["state"] == "tilted"
    ).any()
    keyframe_fraction = len(timeline) / len(sampled_indices) if len(sampled_indices) else 0.0
    results.append({
        "video_name": video_path.name,
        "expected_label": expected_label,
        "predicted_label": predicted_label,
        "correct": expected_label == predicted_label,
        "visible_fraction": visible_fraction,
        "ncc_keyframe_fraction": keyframe_fraction,
        "max_ncc_score": float(ncc_scores.max()) if len(ncc_scores) else np.nan,
        "ncc_keyframe_count": len(timeline),
        "transition_time_seconds": transition_time,
        "tilted_before_flat": tilted_before_flat,
    })

    # Save keyframe-only evidence for later inspection and overlays.
    timelines[f"{expected_label}/{video_path.name}"] = timeline
    clip_output_folder.mkdir(parents=True, exist_ok=True)
    timeline.to_csv(clip_output_folder / "timeline.csv", index=False)
    timeline.to_csv(clip_output_folder / "ncc_keyframes.csv", index=False)

results_df = pd.DataFrame(results)
results_df.to_csv(output_folder / "classification_results.csv", index=False)
display(results_df)
if not results_df.empty:
    display(pd.crosstab(results_df["expected_label"], results_df["predicted_label"], margins=True))
print(f"Saved results to: {output_folder}")


In [ ]:
# Plot normalized minAreaRect angle and NCC evidence for extracted keyframes.
for timeline_name, timeline in timelines.items():
    result = results_df.loc[results_df["video_name"] == Path(timeline_name).name].iloc[0]
    figure, tilt_axis = plt.subplots(figsize=(14, 4))
    tilt_axis.plot(timeline["time_seconds"], timeline["smoothed_angle_deg"], color="black")

    # Draw every configured absolute angle-range boundary on the timeline.
    all_ranges = UPRIGHT_ANGLE_RANGES_DEG + TILTED_ANGLE_RANGES_DEG + FLAT_ANGLE_RANGES_DEG
    range_boundaries = sorted({boundary for bounds in all_ranges for boundary in bounds})
    for boundary in range_boundaries:
        tilt_axis.axhline(boundary, color="grey", linestyle="--", alpha=0.5)

    for state, colour in {"upright": "green", "tilted": "orange", "flat": "red", "uncertain": "grey"}.items():
        rows = timeline["state"] == state
        tilt_axis.scatter(
            timeline.loc[rows, "time_seconds"],
            timeline.loc[rows, "smoothed_angle_deg"],
            s=12,
            color=colour,
            label=state,
        )

    # Plot NCC on a second axis so the visibility threshold is easy to tune.
    ncc_axis = tilt_axis.twinx()
    ncc_axis.plot(timeline["time_seconds"], timeline["ncc_score"], color="royalblue", alpha=0.45, label="NCC score")
    ncc_axis.axhline(NCC_MIN_SIMILARITY, color="royalblue", linestyle=":", label="NCC threshold")
    keyframe_rows = timeline["ncc_keyframe"]
    ncc_axis.scatter(
        timeline.loc[keyframe_rows, "time_seconds"],
        timeline.loc[keyframe_rows, "ncc_score"],
        marker="*",
        s=90,
        color="blue",
        label="NCC keyframe",
    )

    tilt_axis.set_ylim(-95, 95)
    ncc_axis.set_ylim(-1.0, 1.0)
    tilt_axis.set_xlabel("Video time (seconds)")
    tilt_axis.set_ylabel("Normalized minAreaRect long-axis angle (degrees)")
    ncc_axis.set_ylabel("NCC similarity")
    tilt_axis.set_title(f"{timeline_name}\nexpected: {result['expected_label']} | predicted: {result['predicted_label']}")
    tilt_axis.legend(ncol=4, loc="upper left")
    ncc_axis.legend(ncol=3, loc="upper right")
    tilt_axis.grid(alpha=0.2)
    figure.tight_layout()
    plt.show()


In [ ]:
# Generate minAreaRect overlays only for the extracted NCC keyframe images.
overlay_results = []

for expected_label, video_path in test_videos:
    timeline_key = f"{expected_label}/{video_path.name}"
    if timeline_key not in timelines:
        raise KeyError(f"Run the classification cell before overlays: {timeline_key}")

    timeline = timelines[timeline_key]
    clip_output_folder = output_folder / expected_label / video_path.stem
    overlay_folder = clip_output_folder / "ncc_keyframe_overlays"
    overlay_folder.mkdir(parents=True, exist_ok=True)

    selected_rectangle_count = 0
    for _, frame_evidence in timeline.iterrows():
        keyframe_path = Path(frame_evidence["keyframe_path"])
        annotated_frame = cv2.imread(str(keyframe_path), cv2.IMREAD_COLOR)
        if annotated_frame is None:
            raise RuntimeError(f"Could not read extracted NCC keyframe: {keyframe_path}")

        # Recreate the exact classifier mask on this extracted keyframe only.
        bottle_mask = build_bottle_mask(annotated_frame, bottle_config)
        mask_layer = annotated_frame.copy()
        mask_layer[bottle_mask > 0] = (255, 0, 255)
        annotated_frame = cv2.addWeighted(annotated_frame, 0.75, mask_layer, 0.25, 0)
        bottle_contours, _ = cv2.findContours(
            bottle_mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE,
        )

        # Measure every contour retained by bottles_classifier.json.
        rectangle_candidates = []
        for contour in bottle_contours:
            rectangle = cv2.minAreaRect(contour)
            rectangle_width, rectangle_height = rectangle[1]
            if rectangle_width <= 1 or rectangle_height <= 1:
                continue

            # Reject short fragments and merged masks outside the configured length range.
            rectangle_length_px = max(rectangle_width, rectangle_height)
            if not MIN_RECTANGLE_LENGTH_PX <= rectangle_length_px <= MAX_RECTANGLE_LENGTH_PX:
                continue

            angle_deg = normalise_min_area_rect_angle(
                rectangle_width,
                rectangle_height,
                rectangle[2],
            )
            signed_tilt_deg = calculate_signed_tilt(angle_deg)
            absolute_tilt_deg = abs(signed_tilt_deg)
            box_points = cv2.boxPoints(rectangle).astype(np.int32)
            rectangle_candidates.append((absolute_tilt_deg, signed_tilt_deg, angle_deg, rectangle_length_px, box_points))

        # Draw and label the normalized long-axis angle for every accepted bottle.
        for bottle_number, (_, signed_tilt_deg, angle_deg, _, box_points) in enumerate(rectangle_candidates, start=1):
            cv2.drawContours(annotated_frame, [box_points], 0, (180, 180, 180), 1)
            center_x = int(box_points[:, 0].mean())
            center_y = int(box_points[:, 1].mean())
            angle_label = (
                f"B{bottle_number}: angle={angle_deg:.1f} deg | "
                f"tilt={signed_tilt_deg:.1f} deg"
            )
            label_position = (max(0, center_x - 55), max(20, center_y))
            cv2.putText(
                annotated_frame,
                angle_label,
                label_position,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 0, 0),
                3,
            )
            cv2.putText(
                annotated_frame,
                angle_label,
                label_position,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 255, 255),
                1,
            )

        if rectangle_candidates:
            _, selected_signed_tilt, selected_angle, selected_length, selected_box = max(
                rectangle_candidates,
                key=lambda item: item[0],
            )
            state = classify_angle(selected_angle)
            state_colours = {
                "upright": (0, 200, 0),
                "tilted": (0, 165, 255),
                "flat": (0, 0, 255),
                "uncertain": (180, 180, 180),
            }
            box_colour = state_colours[state]
            cv2.drawContours(annotated_frame, [selected_box], 0, box_colour, 3)
            label_x = max(0, int(selected_box[:, 0].min()))
            label_y = max(25, int(selected_box[:, 1].min()) - 8)
            label = (
                f"{state} | angle={selected_angle:.1f} | signed tilt={selected_signed_tilt:.1f} | "
                f"length={selected_length:.1f}px / "
                f"range={MIN_RECTANGLE_LENGTH_PX:.1f}-{MAX_RECTANGLE_LENGTH_PX:.1f}px"
            )
            cv2.putText(
                annotated_frame,
                label,
                (label_x, label_y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                box_colour,
                2,
            )
            selected_rectangle_count += 1

        # Add source-frame and NCC evidence to the keyframe overlay.
        frame_index = int(frame_evidence["frame_index"])
        cv2.putText(
            annotated_frame,
            f"frame {frame_index} | {frame_evidence['time_seconds']:.2f}s",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2,
        )
        ncc_label = (
            f"NCC {frame_evidence['ncc_score']:.3f} | "
            f"{frame_evidence['nearest_ncc_reference']} | KEYFRAME"
        )
        cv2.putText(
            annotated_frame,
            ncc_label,
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (255, 255, 0),
            2,
        )

        overlay_path = overlay_folder / f"{keyframe_path.stem}_min_area_rect.jpg"
        if not cv2.imwrite(str(overlay_path), annotated_frame):
            raise RuntimeError(f"Could not save keyframe overlay: {overlay_path}")
        overlay_results.append({
            "video_name": video_path.name,
            "expected_label": expected_label,
            "frame_index": frame_index,
            "ncc_score": float(frame_evidence["ncc_score"]),
            "state": frame_evidence["state"],
            "overlay_path": overlay_path,
            "selected_rectangle": bool(rectangle_candidates),
        })

    print(
        f"{video_path.name}: saved {len(timeline)} keyframe overlays; "
        f"{selected_rectangle_count} contain a selected rectangle"
    )

overlay_results_df = pd.DataFrame(overlay_results)
display(overlay_results_df)

# Preview the first extracted-keyframe overlays directly in the notebook.
preview_paths = overlay_results_df["overlay_path"].head(8).tolist() if not overlay_results_df.empty else []
if preview_paths:
    figure, axes = plt.subplots(len(preview_paths), 1, figsize=(16, 5 * len(preview_paths)))
    axes = np.atleast_1d(axes)
    for axis, overlay_path in zip(axes, preview_paths):
        overlay_image = cv2.imread(str(overlay_path), cv2.IMREAD_COLOR)
        axis.imshow(cv2.cvtColor(overlay_image, cv2.COLOR_BGR2RGB))
        axis.set_title(Path(overlay_path).name)
        axis.axis("off")
    figure.tight_layout()
    plt.show()
